# DDL: `dbspend360_sql_warehouse_dbu_cost`

Creates the per-warehouse / per-day Databricks DBU cost staging table populated by
`dbspend360_sql_warehouse_dbu_cost_app`.

Keyed on `(warehouse_id, usage_date)`. Both key columns are non-nullable, so the staging MERGE uses plain
`=` equality (no null-safe `<=>` needed, unlike the pipeline staging table).

Filters `system.billing.usage` to `billing_origin_product = 'SQL' AND usage_metadata.warehouse_id IS NOT NULL`.
The second predicate is not redundant: it excludes `JOBS_SERVERLESS_COMPUTE` rows, which bill under the `SQL`
origin product but carry a 100% NULL `warehouse_id` and therefore belong to the jobs tabs, not here.

`warehouse_type` is derived from the SKU name (SERVERLESS / PRO / CLASSIC) rather than joined from
`system.compute.warehouses`, so the staging load stays a single-table scan.

**No `cluster_id` column.** Validation confirmed every `cluster_id` value is NULL for SQL warehouse SKUs, so
carrying the column would add a permanently empty field and a pointless null-safe MERGE predicate.

**No cloud cost.** SQL warehouses run on Databricks-managed compute, so there is no customer-account VM spend
to join from a cloud cost explorer. The DBU cost is the complete cost.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_sql_warehouse_dbu_cost (
  warehouse_id        STRING,
  usage_date          DATE,
  databricks_cost     DOUBLE,
  currency            STRING,
  sku_name            STRING,
  warehouse_type      STRING,
  workspace_id        STRING,
  workspace_covered   BOOLEAN,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_sql_warehouse_dbu_cost")